# Assignment 4 — Part 2: Building Blocks of a Diffusion Model

**Course:** Applied Generative AI  
**Dataset / project context:** CIFAR-10 image generation  
**Questions:** 1–5

This notebook answers the five theory questions in *Assignment 4*, with short explanations, the calculation steps, and small reproducible checks. The code cells can be run from top to bottom. The source practical uses a different frequency parameterization; for Questions 1–2, I follow the question's explicit **maximum period = 10,000** convention.

## Question 1 — Sinusoidal time embedding

**Question.** Given a timestep $t$, describe how the sinusoidal embedding is computed. Write the mathematical formula for the $i$-th dimension of the embedding.

**Answer.** A sinusoidal time embedding maps one scalar timestep into a vector of sine and cosine values at different frequencies. This gives the diffusion network a continuous representation of the noise level or time step.

For an even embedding dimension $d$, one common convention is to use pairs of dimensions. For $j=0,1,\dots,d/2-1$:

$$
\boxed{\operatorname{PE}(t)_{2j}=\sin\!\left(\frac{t}{10000^{2j/d}}\right)}
$$

$$
\boxed{\operatorname{PE}(t)_{2j+1}=\cos\!\left(\frac{t}{10000^{2j/d}}\right)}
$$

Equivalently, with $i=0,\ldots,d-1$, even $i$ use sine and odd $i$ use cosine, with frequency exponent $2\lfloor i/2\rfloor/d$.

**How it works:** first calculate a different frequency for each sine–cosine pair, multiply it by $t$, then concatenate the results into a $d$-dimensional vector. These deterministic features can then be passed into the UNet as time information.

*Convention note:* The provided Module 8 Practical 2 defines a sinusoidal embedding using log-spaced frequencies from 1 to 1000 and $2\pi$ angular scaling. That implementation has the same purpose but produces different numerical values. The formulas above use the explicit max-period-10000 convention requested in Questions 1–2.

In [1]:
import numpy as np

def time_embedding(t, dimension=8, max_period=10000.0):
    """Interleaved sine/cosine encoding, using the Q1 max-period convention."""
    if dimension % 2 != 0:
        raise ValueError("Embedding dimension must be even.")
    j = np.arange(dimension // 2)
    angles = t / (max_period ** (2 * j / dimension))
    result = np.empty(dimension, dtype=float)
    result[0::2] = np.sin(angles)
    result[1::2] = np.cos(angles)
    return result

print("Embedding dimension:", len(time_embedding(1, 8)))
print("Example embedding:", np.round(time_embedding(1, 8), 6))

Embedding dimension: 8
Example embedding: [0.841471 0.540302 0.099833 0.995004 0.01     0.99995  0.001    1.      ]


## Question 2 — Calculate the embedding for $t=1$, $d=8$

**Question.** If the embedding dimension is 8, write out the values of the sinusoidal embedding vector for $t=1$. Assume max period of 10000.

**Work.** Since $d=8$, there are four sine–cosine pairs. Their angles are:

$$
\theta_j = \frac{1}{10000^{2j/8}},\qquad j=0,1,2,3.
$$

| Pair $j$ | Denominator $10000^{2j/8}$ | Angle $\theta_j$ | Dimensions |
|:---:|---:|---:|:---|
| 0 | 1 | 1 | $\sin(1),\cos(1)$ |
| 1 | 10 | 0.1 | $\sin(0.1),\cos(0.1)$ |
| 2 | 100 | 0.01 | $\sin(0.01),\cos(0.01)$ |
| 3 | 1000 | 0.001 | $\sin(0.001),\cos(0.001)$ |

Therefore, with interleaved sine–cosine ordering:

$$
\operatorname{PE}(1) = [\sin 1,\cos 1,\sin 0.1,\cos 0.1,\sin 0.01,\cos 0.01,\sin 0.001,\cos 0.001].
$$

Numerically (rounded to six decimals):

$$
\boxed{[0.841471,\ 0.540302,\ 0.099833,\ 0.995004,\ 0.010000,\ 0.999950,\ 0.001000,\ 1.000000]}
$$

The last cosine is slightly smaller than 1 before rounding. Another common implementation concatenates all sine values first, then all cosine values; it contains the same eight values but in a different order.

In [2]:
t, d, max_period = 1, 8, 10000
j = np.arange(d // 2)
angles = t / (max_period ** (2 * j / d))
print("Angles:", angles)
for idx, a in enumerate(angles):
    print(f"j={idx}: sin({a:g})={np.sin(a):.6f}, cos({a:g})={np.cos(a):.6f}")
print("Embedding:", np.round(time_embedding(t, d, max_period), 6))
assert len(time_embedding(t, d, max_period)) == 8

Angles: [1.    0.1   0.01  0.001]
j=0: sin(1)=0.841471, cos(1)=0.540302
j=1: sin(0.1)=0.099833, cos(0.1)=0.995004
j=2: sin(0.01)=0.010000, cos(0.01)=0.999950
j=3: sin(0.001)=0.001000, cos(0.001)=1.000000
Embedding: [0.841471 0.540302 0.099833 0.995004 0.01     0.99995  0.001    1.      ]


## Question 3 — Relationship to transformer positional encoding

**Question.** How does positional encoding in transformers relate to sinusoidal time embeddings in diffusion models? What is the key difference in how they are used?

**Answer.** Both use sinusoidal functions at different frequencies to turn a scalar index into a vector that a neural network can work with. They are useful because they provide smooth, deterministic information about ordering or progress.

The key difference is **what the scalar represents and how the network uses it**:

- In a **transformer**, positional encoding represents the position of a token in a sequence. It helps self-attention distinguish token order because attention alone does not inherently encode sequence positions.
- In a **diffusion model**, time embedding represents the current diffusion timestep or noise level. The UNet uses it to decide how much noise is present and how to denoise the image at that stage.

So the mathematics is closely related, but transformer positions describe **where a token is**, while diffusion timesteps describe **where an image is in the noising/denoising process**.

## Question 4 — Resolution after three stride-2 downsampling blocks

**Question.** If the input image is of size $64\times64$, and you apply 3 downsampling blocks with stride-2 convolutions, what is the spatial resolution at the bottleneck?

**Work.** Assuming each block halves the height and width (as in a typical padded stride-2 convolution):

$$
64\times64 \longrightarrow 32\times32 \longrightarrow 16\times16 \longrightarrow 8\times8.
$$

The side length is $64/2^3 = 8$.

**Answer: $\boxed{8\times8}$ at the bottleneck.** The number of feature channels is determined separately by the network architecture.

In [3]:
size = 64
for block in range(1, 4):
    size = size // 2
    print(f"After downsampling block {block}: {size} x {size}")
assert size == 8

After downsampling block 1: 32 x 32
After downsampling block 2: 16 x 16
After downsampling block 3: 8 x 8


## Question 5 — UNet output and loss

**Question.** If the model receives an image $x_t$ and timestep $t$, what does the UNet output, and how is that used to compute the loss?

**Answer.** In the **noise-prediction diffusion setup used in our practical work**, the UNet receives a noisy image $x_t$ and a time embedding for $t$. It predicts the Gaussian noise that was added to the clean image. The predicted noise, $\epsilon_\theta(x_t,t)$, has the same shape as the image.

**Step 1 — Construct a noisy training image.** Sample a clean image $x_0$, Gaussian noise $\epsilon\sim\mathcal N(0,I)$, and a timestep $t$. With the course-style signal and noise rates:

$$
x_t = s(t)x_0+n(t)\epsilon.
$$

In our simplified Assignment 4 implementation, $s(t)=\cos(\pi t/2)$ and $n(t)=\sin(\pi t/2)$ for $t\in[0,1]$.

**Step 2 — Predict the noise.**

$$
\hat{\epsilon}=\epsilon_\theta(x_t,t).
$$

**Step 3 — Compute the training loss.** Compare the predicted noise with the actual sampled noise using mean squared error (MSE):

$$
\boxed{\mathcal L(\theta)=\mathbb E_{x_0,t,\epsilon}\left[\left\|\epsilon-\epsilon_\theta(x_t,t)\right\|_2^2\right]}
$$

In code, `F.mse_loss(predicted_noise, noise)` averages the squared elementwise differences across all pixels and channels. Backpropagation updates the UNet weights to make the noise prediction more accurate. During generation, the predicted noise is used repeatedly to estimate and remove noise at successive timesteps.

*Scope:* Some diffusion models instead predict the clean image or a velocity parameterization. For this question, we use noise prediction, which matches the training code developed for this assignment.

In [4]:
import torch
import torch.nn.functional as F

# A small standalone example of the noising equation and MSE loss.
# It demonstrates tensor shapes without requiring a trained UNet.
torch.manual_seed(42)
x0 = torch.randn(2, 3, 32, 32)
t = torch.tensor([0.25, 0.75])
epsilon = torch.randn_like(x0)

signal = torch.cos(t * torch.pi / 2)[:, None, None, None]
noise_rate = torch.sin(t * torch.pi / 2)[:, None, None, None]
xt = signal * x0 + noise_rate * epsilon

# Placeholder prediction for formula verification only (not a trained model)
predicted_noise = torch.zeros_like(xt)
loss = F.mse_loss(predicted_noise, epsilon)

print("Clean image shape:", tuple(x0.shape))
print("Noisy image shape:", tuple(xt.shape))
print("Predicted noise shape:", tuple(predicted_noise.shape))
print("Example MSE loss:", round(loss.item(), 4))
assert xt.shape == predicted_noise.shape == epsilon.shape

Clean image shape: (2, 3, 32, 32)
Noisy image shape: (2, 3, 32, 32)
Predicted noise shape: (2, 3, 32, 32)
Example MSE loss: 0.993


---

### Short wrap-up

The five questions connect three ideas: sinusoidal time encoding tells the model **when** it is denoising, downsampling/upsampling gives the UNet multi-scale image features, and noise-prediction MSE gives it a measurable training objective.

**Reference materials supplied for this assignment:** *Assignment 4*, Section 2 (Questions 1–5); *Module 8 — Energy Based and Diffusion Models* (diffusion slides); *Module 8 Practical 2 — Diffusion Methods*. The numerical convention for Q1–Q2 follows the maximum period explicitly stated in the assignment.